# Batch 3 — Preprocessing (Regularization + Threshold Tuning Setup)

**What Batch 1 and Batch 2 established, and what that means for Batch 3:**

- Batch 1: XGBoost and LightGBM are the (statistically tied) best models,
  ~0.80 PR-AUC, no sampling, full feature set.
- Batch 2: **no imbalance-handling technique beat the no-sampling baseline
  on PR-AUC, for any model** — every technique traded precision for recall
  along the same curve rather than improving the model. **Redundancy
  removal made no difference** except for Random Forest, where it hurt.

So Batch 3 does **not** need to re-derive a feature set or an imbalance
strategy — Batch 2 already answered those questions ("full features, no
sampling" wins). What Batch 3 adds:

1. **Threshold tuning** — the main focus. Batch 2's write-up flagged that
   since PR-AUC is threshold-independent, moving the classification
   threshold on the *existing* baseline models should recover the same
   kind of recall gains sampling was chasing, without sampling's precision
   cost. This notebook doesn't do the tuning itself (that needs trained
   models and belongs in `training.ipynb`) but sets up exactly which models
   and data will be used for it.
2. **Regularization / hyperparameter refinement** — a focused, narrow
   re-search around Batch 1's winning configurations for XGBoost, LightGBM,
   and Logistic Regression, explicitly including L1/L2 regularization
   strength (not swept in Batch 1).
3. **A documented decision on which models continue as finalists.**

This notebook reuses Batch 1's fixed split and encoded feature sets
directly — no new encoding or splitting is needed.


In [1]:
import pandas as pd
import joblib
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
RANDOM_STATE = 42


## 1. Paths & Load Prior Batches' Artifacts

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'Models_Batch_3' else Path.cwd()
BATCH1_DIR = PROJECT_ROOT / 'Models_Batch_1'
BATCH2_DIR = PROJECT_ROOT / 'Models_Batch_2'
BATCH_DIR = PROJECT_ROOT / 'Models_Batch_3'
ARTIFACTS_DIR = BATCH_DIR / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
(ARTIFACTS_DIR / 'models').mkdir(exist_ok=True)

data1 = joblib.load(BATCH1_DIR / 'artifacts' / 'batch1_data.joblib')
batch1_results = pd.read_csv(BATCH1_DIR / 'artifacts' / 'batch1_results.csv')
batch2_results = pd.read_csv(BATCH2_DIR / 'artifacts' / 'batch2_results.csv')
batch2_results['Sampling_Technique'] = batch2_results['Sampling_Technique'].fillna('None')

print("Loaded Batch 1 fixed split + encoders, Batch 1/2 results tables.")
print(f"Train: {data1['X_train_tree'].shape}, Val: {data1['X_val_tree'].shape}, Test: {data1['X_test_tree'].shape}")


Loaded Batch 1 fixed split + encoders, Batch 1/2 results tables.
Train: (35000, 40), Val: (7500, 40), Test: (7500, 40)


## 2. Confirm Batch 2's Conclusion Programmatically

Before building on "full feature set, no sampling wins," verify it directly
from the stored results rather than just asserting it from memory.


In [3]:
# Claim 1 (Track A): within the full feature set, does 'no sampling' beat every sampling technique?
track_a = batch2_results[batch2_results['Feature_Set'] == 'full']
best_sampling_per_model = track_a.loc[track_a.groupby('Model')['Validation_PR_AUC'].idxmax()]
sampling_check = best_sampling_per_model[['Model', 'Sampling_Technique', 'Validation_PR_AUC']].rename(
    columns={'Sampling_Technique': 'Best_Sampling_Technique'})
print("Best sampling technique per model (full feature set):")
print(sampling_check.to_string(index=False))
assert (sampling_check['Best_Sampling_Technique'] == 'None').all(), \
    "Expected 'no sampling' to beat every sampling technique for every model"
print("\nConfirmed: no sampling technique beat the no-sampling baseline, for any model.")

# Claim 2 (Track B): within no sampling, feature-set differences are noise-level EXCEPT for Random Forest
track_b = batch2_results[batch2_results['Sampling_Technique'] == 'None']
print("\nFeature-set comparison (no sampling):")
for model, g in track_b.groupby('Model'):
    spread = g['Validation_PR_AUC'].max() - g['Validation_PR_AUC'].min()
    print(f"  {model:>20}: spread across full/reduced_v1/reduced_v2 = {spread:.4f}")


Best sampling technique per model (full feature set):
             Model Best_Sampling_Technique  Validation_PR_AUC
          LightGBM                    None             0.8008
LogisticRegression                    None             0.7087
               MLP                    None             0.7804
      RandomForest                    None             0.7724
           XGBoost                    None             0.8039

Confirmed: no sampling technique beat the no-sampling baseline, for any model.

Feature-set comparison (no sampling):
              LightGBM: spread across full/reduced_v1/reduced_v2 = 0.0027
    LogisticRegression: spread across full/reduced_v1/reduced_v2 = 0.0001
                   MLP: spread across full/reduced_v1/reduced_v2 = 0.0011
          RandomForest: spread across full/reduced_v1/reduced_v2 = 0.0180
               XGBoost: spread across full/reduced_v1/reduced_v2 = 0.0009


## 3. Select the Finalist Roster for Batch 3

Carrying forward every model from Batch 1/2 would keep re-litigating
models already shown to be non-competitive. Decisions, with reasoning:

| Model | Continue to Batch 3? | Why |
|---|---|---|
| **XGBoost** | Yes — primary focus | Tied-best in Batch 1 (0.8039 PR-AUC), fast to train (55s), most practical leader |
| **LightGBM** | Yes — primary focus | Tied-best in Batch 1 (0.8008 PR-AUC), fastest of the leaders (69s) |
| **Logistic Regression** | Yes — regularization + threshold | Weakest of the "serious" models (0.7087) but the only interpretable one; worth a finer regularization sweep and its own threshold tuning |
| **MLP** | Threshold tuning only | Competitive baseline (0.7804) but Batch 1 already searched it reasonably; no clear regularization lever as clean as L1/L2 for the others |
| **Random Forest** | Threshold tuning only | Behind the leaders (0.7724) and shown in Batch 2 to be sensitive to feature redundancy — not worth further hyperparameter spend, but cheap to include in the threshold comparison |
| **CatBoost** | **No** | Tied with XGBoost/LightGBM on PR-AUC in Batch 1 while taking 22-28x longer to train (1519s vs 55-69s) — poor cost/benefit, decided in Batch 1's own analysis |
| **Decision Tree, SVM, Naive Bayes** | **No** | Clearly behind the pack since Batch 1 (Naive Bayes especially, PR-AUC 0.44); SVM additionally constrained to a training subsample |

**Regularization refinement (Section 4 of `training.ipynb`):** XGBoost,
LightGBM, Logistic Regression only — these are the models with a clean,
well-understood regularization knob (`reg_alpha`/`reg_lambda` for the
boosters, `C`/`penalty` for Logistic Regression) worth sweeping more finely
than Batch 1's exploratory grids did.

**Threshold tuning (Section 5 of `training.ipynb`, the main focus):** all
five continuing models — XGBoost, LightGBM, Logistic Regression, MLP,
Random Forest.


In [4]:
FINALIST_MODELS = ['XGBoost', 'LightGBM', 'LogisticRegression', 'MLP', 'RandomForest']
REGULARIZATION_REFINE_MODELS = ['XGBoost', 'LightGBM', 'LogisticRegression']

print("Threshold-tuning roster:", FINALIST_MODELS)
print("Regularization-refinement roster:", REGULARIZATION_REFINE_MODELS)


Threshold-tuning roster: ['XGBoost', 'LightGBM', 'LogisticRegression', 'MLP', 'RandomForest']
Regularization-refinement roster: ['XGBoost', 'LightGBM', 'LogisticRegression']


## 4. Load Batch 1's Saved Models (Starting Point for Everything Below)

In [5]:
BATCH1_MODEL_FILES = {
    'XGBoost': 'XGBoost.joblib',
    'LightGBM': 'LightGBM.joblib',
    'LogisticRegression': 'LogisticRegression_rob.joblib',  # RobustScaler won in Batch 1
    'MLP': 'MLP.joblib',
    'RandomForest': 'RandomForest.joblib',
}

batch1_models = {}
for name, fname in BATCH1_MODEL_FILES.items():
    batch1_models[name] = joblib.load(BATCH1_DIR / 'artifacts' / 'models' / fname)
    print(f"Loaded {name} from Batch 1: {fname}")


Loaded XGBoost from Batch 1: XGBoost.joblib
Loaded LightGBM from Batch 1: LightGBM.joblib


Loaded LogisticRegression from Batch 1: LogisticRegression_rob.joblib
Loaded MLP from Batch 1: MLP.joblib


Loaded RandomForest from Batch 1: RandomForest.joblib


## 5. Feature Sets Per Model (Unchanged From Batch 1/2)

`tree` (unscaled) for XGBoost/LightGBM/RandomForest, `rob` (RobustScaler)
for Logistic Regression/MLP — same as Batch 1 and Batch 2's confirmed-best
configuration.


In [6]:
FEATURE_SET_MAP = {
    'XGBoost': 'tree', 'LightGBM': 'tree', 'RandomForest': 'tree',
    'LogisticRegression': 'rob', 'MLP': 'rob',
}

for name, fset in FEATURE_SET_MAP.items():
    print(f"{name:>20}: X_train_{fset} shape = {data1[f'X_train_{fset}'].shape}")


             XGBoost: X_train_tree shape = (35000, 40)
            LightGBM: X_train_tree shape = (35000, 40)
        RandomForest: X_train_tree shape = (35000, 40)
  LogisticRegression: X_train_rob shape = (35000, 40)
                 MLP: X_train_rob shape = (35000, 40)


## 6. Save Batch 3 Setup Artifacts

In [7]:
bundle = {
    'X_train_tree': data1['X_train_tree'], 'X_val_tree': data1['X_val_tree'], 'X_test_tree': data1['X_test_tree'],
    'X_train_rob': data1['X_train_rob'], 'X_val_rob': data1['X_val_rob'], 'X_test_rob': data1['X_test_rob'],
    'y_train': data1['y_train'], 'y_val': data1['y_val'], 'y_test': data1['y_test'],
    'FINALIST_MODELS': FINALIST_MODELS,
    'REGULARIZATION_REFINE_MODELS': REGULARIZATION_REFINE_MODELS,
    'FEATURE_SET_MAP': FEATURE_SET_MAP,
}
joblib.dump(bundle, ARTIFACTS_DIR / 'batch3_setup.joblib')
print("Saved:", ARTIFACTS_DIR / 'batch3_setup.joblib')


Saved: C:\Users\Ali Ahmed\OneDrive - Alexandria National University\Desktop\Etax\use_case_1\ML_model\Models_Batch_3\artifacts\batch3_setup.joblib


---
**Next:** `training.ipynb` — (1) a narrow regularization-focused
hyperparameter re-search for XGBoost/LightGBM/Logistic Regression, keeping
whichever configuration (Batch 1's original or Batch 3's refined version)
wins on CV PR-AUC; then (2) threshold tuning for all five finalist models,
using out-of-fold cross-validated probabilities on the training set to
choose each model's threshold honestly (without peeking at validation),
then confirming the result on the held-out validation set.
